# URSim control + feedback demo
Updated: 2026-03-06

This notebook uses the setup that works in your environment:

- **RTDE receive** on **30004**
- **URScript control** on **30002**
- **Dashboard** on **29999**

Goal:
- send a command from Python,
- see the robot move in **Polyscope**,
- and see feedback (`q`, `qd`, `tcp`) in the notebook.


In [ ]:
import socket
import time
import math
from dataclasses import dataclass
from IPython.display import clear_output
import rtde_receive


In [ ]:
@dataclass
class URSimRTDEConfig:
    host: str = "127.0.0.1"
    port_rtde: int = 30004
    port_urscript: int = 30002
    port_dashboard: int = 29999

CFG = URSimRTDEConfig()
CFG


## Port checks

In [ ]:
def tcp_can_connect(host: str, port: int, timeout: float = 2.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True, None
    except OSError as e:
        return False, str(e)

for name, port in {
    "RTDE": CFG.port_rtde,
    "URScript": CFG.port_urscript,
    "Dashboard": CFG.port_dashboard,
}.items():
    ok, err = tcp_can_connect(CFG.host, port)
    print(f"{name:10s} ->", "OK" if ok else f"FAIL ({err})")


## Helpers

In [ ]:
def send_urscript(script: str, host=CFG.host, port=CFG.port_urscript, timeout=2.0):
    with socket.create_connection((host, port), timeout=timeout) as s:
        s.sendall(script.encode("utf-8"))

def dashboard_send(cmd: str, host=CFG.host, port=CFG.port_dashboard, timeout=2.0):
    with socket.create_connection((host, port), timeout=timeout) as s:
        banner = s.recv(4096).decode("utf-8", errors="ignore")
        s.sendall((cmd.strip() + "\n").encode("utf-8"))
        time.sleep(0.05)
        resp = s.recv(4096).decode("utf-8", errors="ignore")
    return (banner + resp).strip()

def urscript_program(lines, name="py_prog"):
    body = "\n  ".join(lines)
    return f"""def {name}():
  {body}
end
{name}()\n"""

def urscript_movej(q, a=0.3, v=0.3):
    return f"movej({list(map(float, q))}, a={a}, v={v})"

def urscript_textmsg(msg: str):
    safe = msg.replace('"', "'")
    return f'textmsg("{safe}")'


## State once

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    print("Connected:", r.isConnected())
    print("q  =", [round(v, 4) for v in r.getActualQ()])
    print("qd =", [round(v, 4) for v in r.getActualQd()])
    print("tcp=", [round(v, 4) for v in r.getActualTCPPose()])
finally:
    r.disconnect()


## Live state monitor
Stop the cell manually when done.

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    while True:
        clear_output(wait=True)
        q = r.getActualQ()
        qd = r.getActualQd()
        tcp = r.getActualTCPPose()
        print("q  =", [round(v, 4) for v in q])
        print("qd =", [round(v, 4) for v in qd])
        print("tcp=", [round(v, 4) for v in tcp])
        time.sleep(0.1)
finally:
    r.disconnect()


## Small move command test

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    q0 = r.getActualQ()
finally:
    r.disconnect()

q_target = q0.copy()
q_target[0] += 0.20

prog = urscript_program([
    urscript_textmsg("move test from notebook"),
    urscript_movej(q_target, a=0.3, v=0.3),
], name="move_test")

send_urscript(prog)
print("Sent moveJ command.")
print("q0      =", [round(v, 4) for v in q0])
print("q_target=", [round(v, 4) for v in q_target])


## Command + feedback together

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    q0 = r.getActualQ()
    q_target = q0.copy()
    q_target[0] += 0.20

    prog = urscript_program([
        urscript_textmsg("command + feedback demo"),
        urscript_movej(q_target, a=0.3, v=0.3),
    ], name="cmd_fb_demo")

    send_urscript(prog)

    t0 = time.time()
    while time.time() - t0 < 8.0:
        clear_output(wait=True)
        q = r.getActualQ()
        qd = r.getActualQd()
        tcp = r.getActualTCPPose()

        print("Command sent.")
        print("q0      =", [round(v, 4) for v in q0])
        print("q_target=", [round(v, 4) for v in q_target])
        print()
        print("q       =", [round(v, 4) for v in q])
        print("qd      =", [round(v, 4) for v in qd])
        print("tcp     =", [round(v, 4) for v in tcp])
        print("q_error =", [round(q_target[i] - q[i], 4) for i in range(6)])

        time.sleep(0.1)
finally:
    r.disconnect()


## Repeated trajectory demo

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    q_center = r.getActualQ()
finally:
    r.disconnect()

q_a = q_center.copy()
q_b = q_center.copy()
q_a[0] -= 0.15
q_b[0] += 0.15

print("q_a =", [round(v, 4) for v in q_a])
print("q_b =", [round(v, 4) for v in q_b])


In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    for step, q_target in enumerate([q_a, q_b, q_a, q_b], start=1):
        prog = urscript_program([
            urscript_textmsg(f"trajectory step {step}"),
            urscript_movej(q_target, a=0.4, v=0.4),
        ], name=f"traj_{step}")
        send_urscript(prog)

        t0 = time.time()
        while time.time() - t0 < 3.0:
            clear_output(wait=True)
            q = r.getActualQ()
            tcp = r.getActualTCPPose()
            print(f"Trajectory step {step}/4")
            print("q_target =", [round(v, 4) for v in q_target])
            print("q        =", [round(v, 4) for v in q])
            print("tcp      =", [round(v, 4) for v in tcp])
            time.sleep(0.1)
finally:
    r.disconnect()


## Stop helper

In [ ]:
print(dashboard_send("stop"))


# Mujoco RTDE Loops

## Function to render in Mujoco
use 50 herz as rendering frequency

In [ ]:
import sys
print(sys.executable)

In [ ]:
from URSim_RTDE_dependencies import URSimRTDEControlFeedback
import time
import mujoco
import matplotlib.pyplot as plt
from IPython.display import clear_output


robot = URSimRTDEControlFeedback()
xml_path = "../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml"


model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)

# Initialize MuJoCo arm pose from keyframe start


renderer = mujoco.Renderer(model, height=480, width=640)
# Create free camera
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE

# Look at robot base / workspace center
cam.lookat[:] = [0.0, 0.0, 0.4]   # x, y, z target point
cam.distance = 2.0                 # zoom out
cam.azimuth = 130                  # rotation around scene
cam.elevation = -20                # tilt angle

joint_qpos_indices = [0, 1, 2, 3, 4, 5]

for _ in range(500):
    fb = robot.receive_feedback()
    q = fb["q"]

    for i in range(6):
        data.qpos[i] = q[i]

    mujoco.mj_forward(model, data)

    renderer.update_scene(data, camera=cam)
    img = renderer.render()

    clear_output(wait=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    time.sleep(0.02) # 50hz

## Now with Comands and rendering - moveJ fast
But send the target commant only once, otherwise it will never accelerate
here first the version that sends many fast updates with moveJ

In [ ]:
import time
import numpy as np
import mujoco
import matplotlib.pyplot as plt
from IPython.display import clear_output
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

xml_path = "../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml"

# -----------------------------
# Frequencies
# -----------------------------
control_hz = 250.0
render_hz = 50.0
render_every = int(control_hz / render_hz)

dt = 1.0 / control_hz

# -----------------------------
# Targets
# -----------------------------
target_q_list = [
    [0.75, -2.20,  1.57, -1.57, -1.57,  0.00],
    [0.95, -1.90,  1.40, -1.30, -1.40,  0.20],
    [0.40, -1.60,  1.20, -1.80, -1.20, -0.30],
    [0.20, -2.00,  1.80, -1.20, -1.80,  0.40],
    [0.85, -1.70,  1.00, -1.60, -1.10,  0.10],
]

# -----------------------------
# Mujoco setup
# -----------------------------
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat[:] = [0.0, 0.0, 0.35]
cam.distance = 2.0
cam.azimuth = 130
cam.elevation = -20

joint_qpos_indices = [0,1,2,3,4,5]

# -----------------------------
# Control loop
# -----------------------------
tol = 0.05

for k, q_target in enumerate(target_q_list):

    next_t = time.perf_counter()
    step = 0

    while True:

        # -----------------------------
        # Send command (250 Hz)
        # -----------------------------
        robot.send_movej(q_target, a=0.4, v=0.4)

        fb = robot.receive_feedback()
        q = fb["q"]
        tcp = fb["tcp_xyz"]

        # -----------------------------
        # Render every 5th iteration (~50 Hz)
        # -----------------------------
        if step % render_every == 0:

            for i, idx in enumerate(joint_qpos_indices):
                data.qpos[idx] = q[i]

            mujoco.mj_forward(model, data)

            renderer.update_scene(data, camera=cam)
            img = renderer.render()

            clear_output(wait=True)
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"Target {k+1}/{len(target_q_list)}")
            plt.show()

            print("q =", [round(v,2) for v in q])
            print("tcp =", [round(v,3) for v in tcp])

        # -----------------------------
        # Check if reached target
        # -----------------------------
        err = np.linalg.norm(np.asarray(q_target) - np.asarray(q))

        if err < tol:
            print("Reached target", k+1)
            time.sleep(0.5)
            break

        # -----------------------------
        # Timing (250 Hz)
        # -----------------------------
        step += 1
        next_t += dt
        sleep_time = next_t - time.perf_counter()
        if sleep_time > 0:
            time.sleep(sleep_time)

robot.disconnect()

## MoveJ one target command


In [ ]:
import time
import numpy as np
import mujoco
import matplotlib.pyplot as plt
from IPython.display import clear_output
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

xml_path = "../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml"

target_q_list = [
    [0.75, -2.20,  1.57, -1.57, -1.57,  0.00],
    [0.95, -1.90,  1.40, -1.30, -1.40,  0.20],
    [0.40, -1.60,  1.20, -1.80, -1.20, -0.30],
    [0.20, -2.00,  1.80, -1.20, -1.80,  0.40],
    [0.85, -1.70,  1.00, -1.60, -1.10,  0.10],
]

render_hz = 10.0
dt_render = 1.0 / render_hz
tol = 0.05
timeout_s = 12.0

model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat[:] = [0.0, 0.0, 0.35]
cam.distance = 2.0
cam.azimuth = 130
cam.elevation = -20

joint_qpos_indices = [0, 1, 2, 3, 4, 5]

for k, q_target in enumerate(target_q_list, start=1):

    # send command ONCE
    robot.send_movej(q_target,a=2.5, v=2.0, textmsg=f"movej target {k}")

    start_t = time.perf_counter()
    next_t = time.perf_counter()

    while True:
        fb = robot.receive_feedback()
        q = fb["q"]
        tcp_xyz = fb["tcp_xyz"]

        for i, idx in enumerate(joint_qpos_indices):
            data.qpos[idx] = q[i]

        mujoco.mj_forward(model, data)
        renderer.update_scene(data, camera=cam)
        img = renderer.render()

        clear_output(wait=True)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"moveJ target {k}/{len(target_q_list)}")
        plt.show()

        err = np.linalg.norm(np.asarray(q_target) - np.asarray(q))
        print("q_target =", [round(v, 2) for v in q_target])
        print("q        =", [round(v, 2) for v in q])
        print("tcp_xyz  =", [round(v, 3) for v in tcp_xyz])
        print("err_norm =", round(err, 4))

        if err < tol:
            print(f"Reached target {k}")
            time.sleep(0.5)
            break

        if time.perf_counter() - start_t > timeout_s:
            print(f"Timeout at target {k}")
            break

        next_t += dt_render
        sleep_time = next_t - time.perf_counter()
        if sleep_time > 0:
            time.sleep(sleep_time)

robot.disconnect()

## servoJ with many command


In [ ]:
import time
import numpy as np
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback(host="127.0.0.1")

q_start = [0.75, -2.20, 1.57, -1.57, -1.57, 0.00]
q_goal  = [0.95, -1.90, 1.40, -1.30, -1.40, 0.20]

hz = 200.0
dt = 1.0 / hz
tol = 0.02
max_step = 0.03
print("dt=", dt)

# move once to start
robot.send_movej(q_start, a=0.4, v=0.4, textmsg="go_start")

# wait until start is reached
while True:
    q_now = np.array(robot.receive_feedback()["q"], dtype=float)
    err = np.linalg.norm(q_now - np.array(q_start, dtype=float))
    print(f"to start | err = {err:.4f}", end="\r")
    if err < 0.05:
        break
    time.sleep(0.01)

print("\nReached start")

# -------------------------
# Start timer
# -------------------------
start_time = time.perf_counter()
step_count = 0

# servoj loop to goal
while True:
    q_cmd, dist = robot.step_toward_joint_target(q_goal, max_step=max_step)
    elapsed = time.perf_counter() - start_time

    print(f"time = {elapsed:.2f}s | dist = {dist:.4f}", end="\r")

    if dist < tol:
        total_time = time.perf_counter() - start_time
        print(f"\nReached goal in {total_time:.3f} seconds ({step_count} steps)")
        break

    robot.send_servoj(
        q_cmd,
        t=dt,
        lookahead_time=0.1,
        gain=300,
    )

    step_count += 1
    time.sleep(dt)

print("Done")

## Including mujoco Rendering

In [ ]:
import time
import numpy as np
import pandas as pd
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback(host="127.0.0.1")

xml_path = "../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml"
mj = robot.mujoco_init_model(xml_path, height=480, width=640)

q_start = [0.75, -2.20, 1.57, -1.57, -1.57, 0.00]
q_goal  = [0.95, -1.90, 1.40, -1.30, -1.40, 0.20]

control_hz = 250.0
render_hz = 10.0

dt = 1.0 / control_hz
render_every = int(control_hz / render_hz)

tol = 0.02
max_speed = 2.0
max_step = max_speed * dt

robot.send_movej(q_start, a=0.4, v=0.4, textmsg="go_start")

while True:
    q_now = np.array(robot.receive_feedback()["q"], dtype=float)
    err = np.linalg.norm(q_now - np.array(q_start))
    if err < 0.05:
        break
    time.sleep(0.05)

print("Reached start")

log = []

start_time = time.perf_counter()
last_loop_time = start_time
step_count = 0
prev_tcp_xyz = None

while True:
    loop_t0 = time.perf_counter()

    recv_t0 = time.perf_counter()
    fb = robot.receive_feedback()
    recv_t1 = time.perf_counter()

    recv_time = recv_t1 - recv_t0

    q = np.asarray(fb["q"], dtype=float)
    qd = np.asarray(fb["qd"], dtype=float)
    tcp_xyz = np.asarray(fb["tcp_xyz"], dtype=float)

    if prev_tcp_xyz is None:
        tcp_delta = np.zeros(3, dtype=float)
        tcp_dist_loop = 0.0
    else:
        tcp_delta = tcp_xyz - prev_tcp_xyz
        tcp_dist_loop = float(np.linalg.norm(tcp_delta))

    prev_tcp_xyz = tcp_xyz.copy()

    robot.mujoco_sync_from_robot(mj, q, qd)

    if step_count % render_every == 0:
        robot.mujoco_render(mj, title="servoJ toward goal")

    q_cmd, dist = robot.step_toward_joint_target(q_goal, max_step=max_step)

    now = time.perf_counter()
    elapsed = now - start_time
    dt_loop = now - last_loop_time
    last_loop_time = now

    log.append({
        "step": step_count,
        "time": elapsed,
        "dt_loop": dt_loop,
        "recv_time": recv_time,
        "dist": dist,

        "q0": q[0], "q1": q[1], "q2": q[2], "q3": q[3], "q4": q[4], "q5": q[5],
        "qd0": qd[0], "qd1": qd[1], "qd2": qd[2], "qd3": qd[3], "qd4": qd[4], "qd5": qd[5],
        "cmd0": q_cmd[0], "cmd1": q_cmd[1], "cmd2": q_cmd[2], "cmd3": q_cmd[3], "cmd4": q_cmd[4], "cmd5": q_cmd[5],

        "tcp_x": tcp_xyz[0],
        "tcp_y": tcp_xyz[1],
        "tcp_z": tcp_xyz[2],

        "tcp_dx": tcp_delta[0],
        "tcp_dy": tcp_delta[1],
        "tcp_dz": tcp_delta[2],
        "tcp_dist_loop": tcp_dist_loop,
    })

    if dist < tol:
        break

    robot.send_servoj(
        q_cmd,
        t=dt,
        lookahead_time=0.1,
        gain=300,
    )

    step_count += 1
    time.sleep(dt)

robot.disconnect()

df = pd.DataFrame(log)

print("Reached goal")
print("Total steps:", step_count)
print("Total time:", time.perf_counter() - start_time)

print("\nTiming statistics")
print("------------------")
print("Mean loop dt:", df["dt_loop"].mean())
print("Std loop dt :", df["dt_loop"].std())
print("Mean recv time:", df["recv_time"].mean())

print("\nTCP motion statistics")
print("---------------------")
print("Mean tcp distance per loop:", df["tcp_dist_loop"].mean())
print("Max  tcp distance per loop:", df["tcp_dist_loop"].max())

print("\nMax joint speed observed")
print("------------------------")
print(df[["qd0","qd1","qd2","qd3","qd4","qd5"]].abs().max())

# Check the runtime with different frequencies
Rund multible runs with different frequencies to evaluate the the loop structure

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from URSim_RTDE_dependencies import URSimRTDEControlFeedback


def run_servoj_experiment(
    robot,
    mj,
    q_start,
    q_goal,
    control_hz=20.0,
    render_hz=5.0,
    tol=0.02,
    max_speed=2.0,          # rad/s
    lookahead_time=0.1,
    gain=300,
    movej_a=0.4,
    movej_v=0.4,
    timeout_s=40.0,
    do_render=True,
    start_reach_tol=0.05,
):
    dt = 1.0 / control_hz
    render_every = max(1, int(round(control_hz / render_hz))) if render_hz > 0 else None
    max_step = max_speed * dt

    robot.send_movej(q_start, a=movej_a, v=movej_v, textmsg="go_start")

    while True:
        q_now = np.asarray(robot.receive_feedback()["q"], dtype=float)
        err = np.linalg.norm(q_now - np.asarray(q_start, dtype=float))
        print(f"to start | err = {err:.4f}", end="\r")
        if err < start_reach_tol:
            break
        time.sleep(0.05)

    print("\nReached start")

    log = []
    prev_tcp_xyz = None
    start_time = time.perf_counter()
    last_loop_time = start_time
    step_count = 0
    reached_goal = False
    timed_out = False

    while True:
        loop_t0 = time.perf_counter()

        recv_t0 = time.perf_counter()
        fb = robot.receive_feedback()
        recv_t1 = time.perf_counter()

        recv_time = recv_t1 - recv_t0

        q = np.asarray(fb["q"], dtype=float)
        qd = np.asarray(fb["qd"], dtype=float)
        tcp_xyz = np.asarray(fb["tcp_xyz"], dtype=float)

        if prev_tcp_xyz is None:
            tcp_delta = np.zeros(3, dtype=float)
            tcp_dist_loop = 0.0
        else:
            tcp_delta = tcp_xyz - prev_tcp_xyz
            tcp_dist_loop = float(np.linalg.norm(tcp_delta))
        prev_tcp_xyz = tcp_xyz.copy()

        robot.mujoco_sync_from_robot(mj, q, qd)

        if do_render and render_every is not None and step_count % render_every == 0:
            robot.mujoco_render(mj, title=f"servoJ | {control_hz:.1f} Hz ctrl | {render_hz:.1f} Hz render")

        q_cmd, dist = robot.step_toward_joint_target(q_goal, max_step=max_step)

        now = time.perf_counter()
        elapsed = now - start_time
        dt_loop = now - last_loop_time
        last_loop_time = now

        log.append({
            "step": step_count,
            "time": elapsed,
            "dt_loop": dt_loop,
            "recv_time": recv_time,
            "dist": dist,
            "q0": q[0], "q1": q[1], "q2": q[2], "q3": q[3], "q4": q[4], "q5": q[5],
            "qd0": qd[0], "qd1": qd[1], "qd2": qd[2], "qd3": qd[3], "qd4": qd[4], "qd5": qd[5],
            "cmd0": q_cmd[0], "cmd1": q_cmd[1], "cmd2": q_cmd[2], "cmd3": q_cmd[3], "cmd4": q_cmd[4], "cmd5": q_cmd[5],
            "tcp_x": tcp_xyz[0], "tcp_y": tcp_xyz[1], "tcp_z": tcp_xyz[2],
            "tcp_dx": tcp_delta[0], "tcp_dy": tcp_delta[1], "tcp_dz": tcp_delta[2],
            "tcp_dist_loop": tcp_dist_loop,
        })

        if dist < tol:
            reached_goal = True
            break

        if elapsed > timeout_s:
            timed_out = True
            break

        robot.send_servoj(
            q_cmd,
            t=dt,
            lookahead_time=lookahead_time,
            gain=gain,
        )

        step_count += 1

        elapsed_loop = time.perf_counter() - loop_t0
        sleep_time = dt - elapsed_loop
        if sleep_time > 0:
            time.sleep(sleep_time)

    df = pd.DataFrame(log)
    if len(df) > 0:
        df["tcp_speed_loop"] = df["tcp_dist_loop"] / df["dt_loop"].clip(lower=1e-9)
        df["qd_norm"] = np.sqrt(
            df["qd0"]**2 + df["qd1"]**2 + df["qd2"]**2 +
            df["qd3"]**2 + df["qd4"]**2 + df["qd5"]**2
        )
        df["cmd_err_norm"] = np.sqrt(
            (df["cmd0"] - df["q0"])**2 + (df["cmd1"] - df["q1"])**2 + (df["cmd2"] - df["q2"])**2 +
            (df["cmd3"] - df["q3"])**2 + (df["cmd4"] - df["q4"])**2 + (df["cmd5"] - df["q5"])**2
        )

    stats = summarize_servoj_run(
        df,
        requested_control_hz=control_hz,
        requested_render_hz=render_hz,
        reached_goal=reached_goal,
        timed_out=timed_out,
        timeout_s=timeout_s,
    )

    return df, stats


def summarize_servoj_run(
    df,
    requested_control_hz,
    requested_render_hz,
    reached_goal,
    timed_out,
    timeout_s,
):
    if len(df) == 0:
        return {
            "requested_control_hz": requested_control_hz,
            "requested_render_hz": requested_render_hz,
            "reached_goal": reached_goal,
            "timed_out": timed_out,
            "timeout_s": timeout_s,
            "num_samples": 0,
        }

    stats = {
        "requested_control_hz": requested_control_hz,
        "requested_render_hz": requested_render_hz,
        "reached_goal": reached_goal,
        "timed_out": timed_out,
        "timeout_s": timeout_s,
        "num_samples": int(len(df)),
        "total_time_s": float(df["time"].iloc[-1]),
        "mean_loop_dt_s": float(df["dt_loop"].mean()),
        "std_loop_dt_s": float(df["dt_loop"].std()),
        "mean_actual_control_hz": float(1.0 / df["dt_loop"].mean()),
        "mean_recv_time_s": float(df["recv_time"].mean()),
        "max_recv_time_s": float(df["recv_time"].max()),
        "start_dist": float(df["dist"].iloc[0]),
        "final_dist": float(df["dist"].iloc[-1]),
        "mean_tcp_dist_loop_m": float(df["tcp_dist_loop"].mean()),
        "max_tcp_dist_loop_m": float(df["tcp_dist_loop"].max()),
        "mean_tcp_speed_mps": float(df["tcp_speed_loop"].mean()),
        "max_tcp_speed_mps": float(df["tcp_speed_loop"].max()),
        "mean_qd_norm": float(df["qd_norm"].mean()),
        "max_qd_norm": float(df["qd_norm"].max()),
        "mean_cmd_err_norm": float(df["cmd_err_norm"].mean()),
        "max_cmd_err_norm": float(df["cmd_err_norm"].max()),
    }

    for j in range(6):
        stats[f"max_abs_qd{j}"] = float(df[f"qd{j}"].abs().max())

    return stats


def print_servoj_stats(stats):
    print("Run statistics")
    print("---------------------------")
    for k, v in stats.items():
        if isinstance(v, float):
            print(f"{k}: {v:.6f}")
        else:
            print(f"{k}: {v}")


def compare_servoj_runs(results):
    rows = []
    for label, _, stats in results:
        row = {"label": label}
        row.update(stats)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
import numpy as np
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback(host="127.0.0.1")

xml_path = "../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml"
mj = robot.mujoco_init_model(xml_path, height=480, width=640)

q_start = [0.75, -2.20, 1.57, -1.57, -1.57, 0.00]
q_goal  = [0.95, -1.90, 1.40, -1.30, -1.40, 0.20]

df_10_5, stats_10_5 = run_servoj_experiment(
    robot, mj,
    q_start=q_start,
    q_goal=q_goal,
    control_hz=10.0,
    render_hz=5.0,
    max_speed=2.0,
    do_render=True,
)

# print_servoj_stats(stats_10_5)

In [ ]:
results = []

for control_hz, render_hz in [(50.0, 10.0), (250.0, 10.0), (50.0, 20.0), (250.0, 50.0)]:
    print(f"\nRunning control_hz={control_hz}, render_hz={render_hz}")
    df_run, stats_run = run_servoj_experiment(
        robot, mj,
        q_start=q_start,
        q_goal=q_goal,
        control_hz=control_hz,
        render_hz=render_hz,
        max_speed=2.0,
        do_render=False,
        timeout_s=20.0,
    )
    label = f"ctrl_{control_hz}_render_{render_hz}"
    results.append((label, df_run, stats_run))


In [ ]:
df_compare = compare_servoj_runs(results)
df_compare[[
    "label",
    "requested_control_hz",
    "requested_render_hz",
    "reached_goal",
    "total_time_s",
    "mean_actual_control_hz",
    "mean_recv_time_s",
    "mean_tcp_dist_loop_m",
    "mean_tcp_speed_mps",
    "max_tcp_speed_mps",
    "mean_qd_norm",
    "final_dist",
]]

# Systematic loop

In [ ]:
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback(host="127.0.0.1")

q_start = [0.75, -2.20, 1.57, -1.57, -1.57, 0.00]
q_goal  = [0.95, -1.90, 1.40, -1.30, -1.40, 0.20]

df, stats = robot.run_step_toward_joint_target_experiment(
    q_start=q_start,
    q_goal=q_goal,
    control_hz=100.0,
    tol=0.01,
    timeout_s=20.0,
    max_speed=2.0,
    lookahead_time=0.1,
    gain=300,
)

keys = ["total_time_s",
        "---",
        "requested_control_hz", 
        "mean_loop_dt_true_s", 
        "mean_loop_hz_true",
        "---",
        "mean_recv_time_s",
        "mean_policy_time_s",
        "mean_send_time_s",
        "mean_log_time_s",
        "---",
        "mean_tcp_speed_mps"]

robot.print_stats_keys(stats, keys)

In [ ]:
import pandas as pd
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback(host="127.0.0.1")

q_start = [0.75, -2.20, 1.57, -1.57, -1.57, 0.00]
q_goal  = [0.95, -1.90, 1.40, -1.30, -1.40, 0.20]

test_hz = [10.0, 50.0, 100.0, 250.0, 500.0]

rows = []
runs = {}

for hz in test_hz:
    # print(f"\nRunning requested control_hz = {hz}")

    df, stats = robot.run_step_toward_joint_target_experiment(
        q_start=q_start,
        q_goal=q_goal,
        control_hz=hz,
        tol=0.02,
        timeout_s=200.0,
        max_speed=2.0,
        lookahead_time=0.1,
        gain=300,
    )

    runs[hz] = {"df": df, "stats": stats}

    requested_hz = float(stats["requested_control_hz"])
    true_hz = float(stats["mean_loop_hz_true"])
    true_inferred_hz = stats["true_inferred_frequency_hz"]

    diff_hz = true_hz - requested_hz
    diff_pct = 100.0 * diff_hz / requested_hz

    rows.append({
        "requested_hz": round(float(stats["requested_control_hz"]), 1),
        "true_hz": round(float(stats["mean_loop_hz_true"]), 1),
        "true_inferred_hz": round(float(stats["true_inferred_frequency_hz"]), 1),
        "diff_hz": round(diff_hz, 1),
        "diff_pct": round(diff_pct, 1),
        "total_loop_wall_time_s": round(stats["total_loop_wall_time_s"], 3),
    })

df_compare = pd.DataFrame(rows)
df_compare

## Loop, then render


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation

from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

q_start = [0.75, -2.20, 1.57, -1.57, -1.57, 0.00]
q_goal  = [0.95, -1.90, 1.40, -1.30, -1.40, 0.20]


df, stats = robot.run_step_toward_joint_target_experiment(
    q_start=q_start,
    q_goal=q_goal,
    control_hz=100.0,
    tol=0.01,
    timeout_s=20.0,
)

keys = ["total_time_s",
        "---",
        "requested_control_hz", 
        "mean_loop_dt_true_s", 
        "mean_loop_hz_true",
        "true_inferred_frequency_hz",
        "---",
        "mean_recv_time_s",
        "mean_policy_time_s",
        "mean_send_time_s",
        "mean_log_time_s",
        "---",
        "mean_tcp_speed_mps"]

robot.print_stats_keys(stats, keys)



In [ ]:
mj = robot.mujoco_init_model(
    xml_path="../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml",
    height=480,
    width=640,
    cam_lookat=(0.0, 0.0, 0.45),
    cam_distance=2,
    cam_azimuth=140,
    cam_elevation=-25,
)
robot.mujoco_replay_logged_loop_in_notebook(
    mj=mj,
    df=df,
    target_render_hz=25.0,
    real_time_scale=1.0,
    original_total_time_s=stats["total_time_s"],
)

In [ ]:
import os
import jax
import jax.numpy as jnp
from flax import serialization
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
import pickle

def load_local_policy(policy_path, env):
    # ---- load params
    params_file = os.path.join(policy_path, "params.pkl")

    with open(params_file, "rb") as f:
        params = pickle.load(f)

    # ---- rebuild network (same as training!)
    ppo_net = ppo_networks.make_ppo_networks(
        observation_size=env.observation_size,
        action_size=env.action_size,
        preprocess_observations_fn=running_statistics.normalize,
    )

    make_policy = ppo_networks.make_inference_fn(ppo_net)

    # params structure: (normalizer, policy, value)
    raw_policy = make_policy((params[0], params[1]), deterministic=True)
    @jax.jit
    def policy_obs_only(obs):
        key = jax.random.PRNGKey(0)
        action, _extras = raw_policy(obs, key)
        return action

    return policy_obs_only

In [ ]:
from mujoco_playground import registry
policy_path = "../../evaluation/downloaded_policies/reach_policy"

env = registry.load("UR10PickCube")

policy_fn = load_local_policy(policy_path, env)
print(type(policy_fn))

# Check out Observation state of UR10PickCube
I need to understand how to build and concatenate the observation state. What to get from the interface and what to get from other sources and where to set the cube position and the target position

In [ ]:
print(env.observation_size)
print(env.action_size)

In [ ]:
dummy_obs = jnp.zeros(env.observation_size, dtype=jnp.float32)
action = policy_fn(dummy_obs)

print(action)
print(action.shape)


In [ ]:
print(type(env))
print(dir(env))

In [ ]:
state = env.reset(jax.random.PRNGKey(0))
print(state.obs.shape)
print(state.obs)

## Observation built for RTDE
use the receive from rtde and build up a full observation that matches the 

In [ ]:
import numpy as np
import importlib
import URSim_RTDE_dependencies
importlib.reload(URSim_RTDE_dependencies)

from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

obs, parts, fb = robot.get_obs_rtde(
    return_parts=True,
    return_feedback=True,
)

print("\n================ PART LENGTH CHECK ================")
total = 0
for name, arr in parts.items():
    n = len(arr)
    total += n
    print(f"{name:32s} len={n:2d}  values={np.round(arr, 4)}")

print("\nTotal from parts:", total)
print("obs.shape:", obs.shape)

In [ ]:
import numpy as np
import importlib
import URSim_RTDE_dependencies
importlib.reload(URSim_RTDE_dependencies)

from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

obs, parts, fb = robot.get_obs_rtde(
    return_parts=True,
    return_feedback=True,
)

obs = np.asarray(obs, dtype=np.float32)

# 2) Add batch dimension
obs_batch = obs[None, :]

# --- From RTDE feedback (ground truth robot state) ---
q_fb = np.asarray(fb["q"], dtype=np.float32)
tcp_fb = np.asarray(fb["tcp_xyz"], dtype=np.float32)

# --- From observation parts (what policy actually sees) ---
q_obs = parts["qpos"][:6]        # first 6 = robot joints
qd_obs = parts["qvel"][:6]

grip_pos_obs = parts["gripper_pos"]   # tcp_xyz inside obs
obj_pos_obs = parts["box_pos"]        # cube position
target_pos = parts["target_pos"]

# 3) Run policy
action = robot._policy_fn(obs_batch)
print("---- ROBOT JOINTS ----")
print("q (fb)  :", np.round(q_fb, 5))
print("q (obs) :", np.round(q_obs, 5))

print("\n---- TCP POSITION ----")
print("tcp_xyz (fb)  :", np.round(tcp_fb, 5))
print("tcp_xyz (obs) :", np.round(grip_pos_obs, 5))

print("\n---- OBJECT ----")
print("object pos (obs) :", np.round(obj_pos_obs, 5))
print("target pos (obs) :", np.round(target_pos, 5))

print("\n---- ACTION (policy output) ----")
print("action:", np.round(action, 5))
print("action shape:", action.shape)

action_np = np.asarray(action, dtype=np.float32)

# remove batch dim: [1, 7] -> [7]
if action_np.ndim == 2 and action_np.shape[0] == 1:
    action_np = action_np[0]

action_arm = action_np[:6]
action_gripper = action_np[6]

step_vec = action_arm - q_obs
step_size = np.linalg.norm(step_vec)

print("\n---- ACTION STEP ----")
print("action_arm     :", np.round(action_arm, 5))
print("action_gripper :", np.round(action_gripper, 5))
print("step_vec       :", np.round(step_vec, 5))
print("step_size      :", float(step_size))

In [ ]:
# ============================================================
# Track whether TCP world position moves closer to target world position
# ============================================================

import numpy as np
import importlib
import URSim_RTDE_dependencies
importlib.reload(URSim_RTDE_dependencies)

from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

# persistent variable across notebook executions
if "prev_tcp_target_dist" not in globals():
    prev_tcp_target_dist = None

# ---- get current obs + feedback ----
obs, parts, fb = robot.get_obs_rtde(
    return_parts=True,
    return_feedback=True,
)

# ---- current TCP world position from RTDE ----
tcp_pos_world = np.asarray(fb["tcp_xyz"], dtype=np.float32)

# ---- reconstruct target world position from obs parts ----
# obj_pos = gripper_pos + (obj_pos - gripper_pos)
obj_pos_world = (
    np.asarray(parts["gripper_pos"], dtype=np.float32)
    + np.asarray(parts["obj_to_gripper"], dtype=np.float32)
)

# target_pos = obj_pos + (target_pos - obj_pos)
target_pos_world = (
    obj_pos_world
    + np.asarray(parts["obj_to_target"], dtype=np.float32)
)

# ---- current distance ----
tcp_target_dist = np.linalg.norm(tcp_pos_world - target_pos_world)
# update memory for next execution


print("\n================ TCP -> TARGET DISTANCE =================\n")
print("tcp_pos_world    :", np.round(tcp_pos_world, 5))
print("target_pos_world :", np.round(target_pos_world, 5))
print("dist_now         :", float(tcp_target_dist))

if prev_tcp_target_dist is None:
    print("dist_prev        : None")
    print("change           : first measurement")
else:
    dist_change = prev_tcp_target_dist - tcp_target_dist

    print("dist_prev        :", float(prev_tcp_target_dist))
    print("dist_change      :", float(dist_change))

    if dist_change > 0:
        print("status           : MOVED CLOSER")
    elif dist_change < 0:
        print("status           : MOVED FARTHER AWAY")
    else:
        print("status           : NO CHANGE")

print("\n========================================================\n")
prev_tcp_target_dist = float(tcp_target_dist)


# Make full loop with trained policy


In [2]:
import numpy as np
import importlib
import URSim_RTDE_dependencies
importlib.reload(URSim_RTDE_dependencies)

from URSim_RTDE_dependencies import URSimRTDEControlFeedback
robot = URSimRTDEControlFeedback()

q_start = [0, -1.7, 2.25, -2.15, -1.5, -1.5]
q_goal  = [0.1, -1.90, 1.40, -1.30, -1.40, 0.20]

robot.move_to_start(q_start, debug_print=True)

# initialize from CURRENT manually positioned arm
df, stats = robot.run_policy_loop(
    q_goal=q_goal,
    control_hz=50.0,
    timeout_s=10.0,
    action_scale=0.01,
    target_pos=[0.1, 0.2, 0.3],
    debug_print=True,
)



Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
[INFO] Loaded policy from: ../../evaluation/downloaded_policies/reach_policy
[INFO] Env: UR10PickCube
[INFO] Obs dim: 63, Act dim: 7
Moving to start pose with movej, a=2.5, v=2.0...
Pre-positioning: qpos=[ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] err=0.00020.0353
Reached start pose in 3.05 seconds
[ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ]
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ] | tcp: [-0.38 -0.14  0.28] 
q: [ 0.   -1.7   2.24 -2.16 -1.49 -1.5 ] | tcp: 

In [ ]:
keys = ["total_time_s",
        "---",
        "requested_control_hz", 
        "mean_loop_dt_true_s", 
        "mean_loop_hz_true",
        "true_inferred_frequency_hz",
        "---",
        "mean_obs_time_s",
        "mean_policy_time_s",
        "mean_send_time_s",
        "mean_log_time_s",
        "---",
        "mean_tcp_speed_mps"]

robot.print_stats_keys(stats, keys)

In [ ]:
mj = robot.mujoco_init_model(
    xml_path="../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml",
    height=480,
    width=640,
    cam_lookat=(0.0, 0.0, 0.45),
    cam_distance=2,
    cam_azimuth=140,
    cam_elevation=-25,
)
robot.mujoco_replay_logged_loop_in_notebook(
    mj=mj,
    df=df,
    target_render_hz=25.0,
    real_time_scale=1.0,
    original_total_time_s=stats["total_time_s"],
)